In [2]:
pip install pandas scikit-learn monpa xgboost lightgbm matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [4]:
# filename: 01_preprocessing.py
import pandas as pd
import multiprocessing
from sklearn.model_selection import train_test_split
import monpa
from utils import monpa_cut_wrapper, clean_text  # 從 utils 匯入函式

# ==========================================
# 設定區
# ==========================================
TRAIN_FILE = 'TrainingData.csv'
TEST_FILE = 'TestingData.csv'
OUTPUT_TRAIN_PKL = 'train_seg.pkl'
OUTPUT_TEST_PKL = 'test_seg.pkl'
NUM_CORES = 12  # 設定使用的核心數

if __name__ == '__main__':
    # 1. 讀取與合併資料
    print("1. 讀取資料中...")
    try:
        df1 = pd.read_csv(TRAIN_FILE)
        df2 = pd.read_csv(TEST_FILE)
    except FileNotFoundError:
        print("錯誤：找不到 CSV 檔案。")
        exit()

    # 合併兩份資料
    df = pd.concat([df1, df2], ignore_index=True)
    print(f"   合併後資料總筆數: {len(df)}")

    # 2. 資料清洗與特徵組合
    print("2. 進行資料清洗與特徵組合...")
    df['Column2'] = df['Column2'].fillna('')
    df['Column3'] = df['Column3'].fillna('')
    # 組合標題 (Column2) + 內文 (Column3)
    df['raw_text'] = df['Column2'] + " " + df['Column3']
    df['raw_text'] = df['raw_text'].apply(clean_text)

    # 準備資料
    texts = df['raw_text'].tolist()
    labels = df['Column1'].values

    # 3. 多核心斷詞
    print(f"3. 啟動 {NUM_CORES} 核進行 Monpa 斷詞 (請耐心等候)...")
    
    # 先初始化 monpa (載入模型)
    _ = monpa.cut("初始化")

    # 使用 Pool 平行運算
    # 注意：這裡使用了 utils.py 裡的函式，解決 AttributeError
    with multiprocessing.Pool(processes=NUM_CORES) as pool:
        cut_results = pool.map(monpa_cut_wrapper, texts)
    
    df['seg_text'] = cut_results
    
    # 4. 分割資料集 (80% 訓練, 20% 測試)
    print("4. 分割資料集 (80% 訓練, 20% 測試)...")
    X_train, X_test, y_train, y_test = train_test_split(
        df['seg_text'], 
        labels, 
        test_size=0.2, 
        random_state=42, 
        stratify=labels
    )

    # 5. 儲存結果 (.pkl 檔讀寫速度快且保留格式)
    print(f"5. 儲存至 {OUTPUT_TRAIN_PKL} 與 {OUTPUT_TEST_PKL}...")
    pd.DataFrame({'text': X_train, 'label': y_train}).to_pickle(OUTPUT_TRAIN_PKL)
    pd.DataFrame({'text': X_test, 'label': y_test}).to_pickle(OUTPUT_TEST_PKL)
    
    print("前處理完成！請執行 02_training.py")

ModuleNotFoundError: No module named 'utils'